# BIG2015 stratified subsample extraction

Extracts a fixed, seeded, per-family subsample of `.bytes` files from the Microsoft Malware
Classification Challenge (BIG 2015) `train.7z`, per [ADR-0003](https://github.com/Tanishk75/MalMap/blob/master/docs/adr/0003-big2015-stratified-subsample.md) and the frozen protocol at [`protocols/big2015_sampling.md`](https://github.com/Tanishk75/MalMap/blob/master/protocols/big2015_sampling.md).

**Before running this notebook:**
1. Add the BIG2015 competition data as a Kaggle input (search *Microsoft Malware Classification Challenge (BIG 2015)* under "Add Data", or attach the competition directly if you've accepted its rules).
2. Confirm `protocols/big2015_sampling.md` in the repo is still the version you intend to run against — this notebook enforces it, it does not decide it. If you need different numbers, edit that file in the repo, commit it, and re-run this notebook against the new commit.

**What this notebook does NOT do:** extract `.asm` listings (ADR-0003), extract anything outside the frozen sample list, or silently change the sampling protocol if live counts disagree with it.

## 1. Clone the repo and install dependencies
Single source of truth for the sampling constants and the registry-building code lives in `src/`, not duplicated in this notebook.

In [ ]:
!git clone --depth 1 https://github.com/Tanishk75/MalMap.git /kaggle/working/MalMap

import sys

sys.path.insert(0, "/kaggle/working/MalMap")



!pip install -q py7zr

In [ ]:
import glob

import json as jsonlib

import shutil

from pathlib import Path



import numpy as np

import pandas as pd

import py7zr



from src.config import BIG2015_SAMPLE_SEED, BIG2015_SAMPLES_PER_FAMILY

from src.data.registry import BIG2015_CLASS_NAMES, build_registry, make_splits



print("seed:", BIG2015_SAMPLE_SEED, "cap per family:", BIG2015_SAMPLES_PER_FAMILY)

## 2. Locate the attached competition input
Auto-discovers `train.7z` and `trainLabels.csv` under `/kaggle/input` rather than hardcoding a dataset slug, since Kaggle assigns that slug per-attachment.

In [ ]:
train_7z_candidates = glob.glob("/kaggle/input/**/train.7z", recursive=True)

labels_candidates = glob.glob("/kaggle/input/**/trainLabels.csv", recursive=True)



assert train_7z_candidates, "train.7z not found under /kaggle/input -- attach the BIG2015 competition data first"

assert labels_candidates, "trainLabels.csv not found under /kaggle/input -- attach the BIG2015 competition data first"



TRAIN_7Z = Path(train_7z_candidates[0])

LABELS_CSV = Path(labels_candidates[0])

print("train.7z:", TRAIN_7Z)

print("trainLabels.csv:", LABELS_CSV)

## 3. Verify published counts against the live labels file
`protocols/big2015_sampling.md` records the per-family counts from memory. This cell recomputes them from the actual attached file and stops if they disagree -- a silent mismatch here would make the Development Plan's "extracted set matches the protocol exactly" exit-gate check meaningless.

In [ ]:
EXPECTED_COUNTS = {

    "Ramnit": 1541, "Lollipop": 2478, "Kelihos_ver3": 2942, "Vundo": 475,

    "Simda": 42, "Tracur": 751, "Kelihos_ver1": 398, "Obfuscator.ACY": 1228, "Gatak": 1013,

}



labels = pd.read_csv(LABELS_CSV)

labels["family_label"] = labels["Class"].map(BIG2015_CLASS_NAMES)

live_counts = labels["family_label"].value_counts().to_dict()



mismatches = {

    fam: (EXPECTED_COUNTS[fam], live_counts.get(fam))

    for fam in EXPECTED_COUNTS

    if EXPECTED_COUNTS[fam] != live_counts.get(fam)

}



print(pd.DataFrame({"expected": EXPECTED_COUNTS, "live": live_counts}))



if mismatches:

    raise AssertionError(

        f"live per-family counts disagree with protocols/big2015_sampling.md: {mismatches}. "

        "Fix the protocol file in the repo first -- do not proceed with a stale table."

    )

print("Live counts match the frozen protocol.")

## 4. Select the frozen sample list
Exact procedure from `protocols/big2015_sampling.md`: sorted family order, sorted ids within a family, one `RandomState` stream seeded once (never re-seeded per family), first `min(cap, available)` after shuffling.

In [ ]:
rng = np.random.RandomState(BIG2015_SAMPLE_SEED)

selected_ids = []

selection_table = []



for family in sorted(EXPECTED_COUNTS):

    family_ids = np.sort(labels.loc[labels["family_label"] == family, "Id"].to_numpy())

    rng.shuffle(family_ids)

    n_take = min(BIG2015_SAMPLES_PER_FAMILY, len(family_ids))

    taken = family_ids[:n_take]

    selected_ids.extend(taken.tolist())

    selection_table.append({"family": family, "available": len(family_ids), "selected": n_take})



selection_df = pd.DataFrame(selection_table)

print(selection_df)

print("Total selected:", len(selected_ids))

## 5. Safety ceiling check before extracting anything
Reads per-member uncompressed sizes from the archive index -- no extraction yet -- and aborts if the projected total exceeds the 15GB ceiling in the protocol.

Matches by **basename**, not full archive path: `train.7z` may store members flat (`<id>.bytes`) or under a subdirectory (e.g. `train/<id>.bytes`) -- this notebook doesn't assume which, since that can only be confirmed by reading this specific archive's index.

In [ ]:
SAFETY_CEILING_BYTES = 15 * 1024**3

target_basenames = {f"{i}.bytes" for i in selected_ids}



with py7zr.SevenZipFile(TRAIN_7Z, mode="r") as archive:

    members = archive.list()



by_basename = {}

for m in members:

    name = Path(m.filename).name

    if name in by_basename:

        raise AssertionError(f"duplicate basename {name!r} in archive -- basename matching is unsafe here")

    by_basename[name] = m



missing_targets = target_basenames - set(by_basename)

if missing_targets:

    sample_names = [m.filename for m in members[:5]]

    raise AssertionError(

        f"{len(missing_targets)} selected ids have no matching .bytes member in train.7z "

        f"(e.g. {sorted(missing_targets)[:5]}). Sample of actual archive member names, to check "

        f"for a naming/path mismatch: {sample_names}"

    )



selected_members = {by_basename[name].filename for name in target_basenames}

projected_bytes = sum(by_basename[name].uncompressed for name in target_basenames)

print(f"Projected extraction size: {projected_bytes / 1024**3:.2f} GB "

      f"(ceiling {SAFETY_CEILING_BYTES / 1024**3:.0f} GB)")



if projected_bytes > SAFETY_CEILING_BYTES:

    raise AssertionError(

        "Projected size exceeds the safety ceiling. Lower BIG2015_SAMPLES_PER_FAMILY in "

        "src/config.py, re-freeze protocols/big2015_sampling.md, and re-run -- do not extract anyway."

    )

## 6. Extract exactly the selected `.bytes` files
Targeted extraction, not a full unpack -- `.asm` listings are never touched.

**Caveat:** if `train.7z` uses solid compression across large blocks, `py7zr` may still need to decompress a whole block to reach one requested member, so this can be slower than the target count suggests. If it stalls for an impractically long time, that is itself a finding worth recording in `protocols/big2015_sampling.md` rather than silently working around.

In [ ]:
OUT_DIR = Path("/kaggle/working/big2015_subsample")

BYTES_DIR = OUT_DIR / "bytes"

BYTES_DIR.mkdir(parents=True, exist_ok=True)



with py7zr.SevenZipFile(TRAIN_7Z, mode="r") as archive:

    archive.extract(path=str(BYTES_DIR), targets=sorted(selected_members))



# Extraction preserves each member's archive-internal path under BYTES_DIR (flat or nested);

# flatten so the layout matches bytes/<id>.bytes regardless of how train.7z stored it.

for extracted_path in list(BYTES_DIR.rglob("*.bytes")):

    dest = BYTES_DIR / extracted_path.name

    if extracted_path != dest:

        shutil.move(str(extracted_path), str(dest))

for leftover_dir in sorted((p for p in BYTES_DIR.iterdir() if p.is_dir()), reverse=True):

    leftover_dir.rmdir()



extracted = sorted(p.name for p in BYTES_DIR.glob("*.bytes"))

assert len(extracted) == len(target_basenames), (

    f"extracted {len(extracted)} files, expected {len(target_basenames)}"

)

print(f"Extracted {len(extracted)} .bytes files to {BYTES_DIR}")

## 7. Write the subsetted `trainLabels.csv` and build the registry
Matches the layout `src.data.registry.build_registry("big2015", ...)` expects: `bytes/<id>.bytes` plus a `trainLabels.csv` at the same root.

In [ ]:
subset_labels = labels[labels["Id"].isin(selected_ids)][["Id", "Class"]]

subset_labels.to_csv(OUT_DIR / "trainLabels.csv", index=False)

print(f"Wrote {len(subset_labels)} rows to {OUT_DIR / 'trainLabels.csv'}")



registry = build_registry("big2015", str(OUT_DIR))

registry = make_splits(registry)

print(registry["family_label"].value_counts())

print(registry["split"].value_counts())

## 8. Copy the registry and family map into the export directory
`build_registry`/`make_splits` persist under the cloned repo's `data_cache/` by default; copy them alongside the extracted files so the whole `big2015_subsample/` directory is self-contained.

In [ ]:
repo_data_cache = Path("/kaggle/working/MalMap/data_cache")

shutil.copy(repo_data_cache / "registry_big2015.csv", OUT_DIR / "registry_big2015.csv")

shutil.copy(repo_data_cache / "family_to_id_big2015.json", OUT_DIR / "family_to_id_big2015.json")



print("Final export layout:")

for p in sorted(OUT_DIR.rglob("*")):

    if p.is_file() and p.parent == OUT_DIR:

        print(" ", p.relative_to(OUT_DIR))

print(f"  bytes/  ({len(list(BYTES_DIR.glob('*.bytes')))} files)")

## 9. Export before the session ends
Per ADR-0006, nothing under `/kaggle/working` survives past the session on its own. Use **"Save Version"** with output files enabled (Kaggle keeps `/kaggle/working` as the version's output dataset), or explicitly zip and download the cell below.

Download `big2015_subsample.zip`, unpack it, and point `build_registry("big2015", "<unpacked_path>")` at it from a local or Colab environment -- the registry and split files inside are already built, so this mainly re-validates the layout.

In [ ]:
shutil.make_archive("/kaggle/working/big2015_subsample", "zip", root_dir=OUT_DIR)

print("Wrote /kaggle/working/big2015_subsample.zip -- download it from the notebook's output pane.")